# AI4GSD Demo
------------------------------------------
## Smartphone videos
------------------------------------------
#### Table of contents
#### 0. Installing requried packages
#### 1. Importing required packages
#### 2. Defining working directory
#### 3. Extracting or copying your videos/photos to desired folder
#### 4. Obtaining scales from AI, drone flight data, or manual measurements
#### 5. Obtaining segmentation of anthropogenic objects
#### 6. Obtaining grain size data
#### 7. Applying quality control

##### *** Please contact Yunxiang Chen, yunxiang.chen@pnnl.gov for reporting errors

------------------------------------------
### 0. Installing requried packages

#### 0a. Mount workshop share drive to your personal google drive.

In [13]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


### 0b. Install requried packages

In [14]:
%pip install -q numpy==2.2.6 pandas==2.2.3 matplotlib==3.10.6 opencv-python==4.12.0.88
%pip install -q torch torchvision ultralytics==8.3.186
%pip install -q natsort pyproj PyExifTool piexif imutils shapely psutil

### 0c. Define Google drive parent path and add it to system.

In [15]:
import sys
from pathlib import Path
WorkDir = Path("/content/drive/MyDrive/AI4GSD")
sys.path.insert(0,str(WorkDir))

exiftool = WorkDir/"AI4GSD/exes/exiftool_linux/Image-ExifTool-13.34/exiftool" # Enable wrting permission in Google drive.
exiftool.chmod(0o755)

### 1. Importing required packages

In [16]:
import AI4GSD

### 2. Defining working directory

In [17]:
AI4GSD.pcinfo()
foldername = '20250521_YC_WalkSmartphoneRoadTest'                          # Parent folder where you save photos/videos.
PhotoDir = str(WorkDir / 'Demo_Data' / foldername)                         # AI4GSD use string path.

Machine name: 1821e9df010d
Operating system: Ubuntu 24.04.5 LTS
Architecture: x86_64
Processor: Intel(R) Xeon(R) CPU @ 2.20GHz
Reported CPU frequency: 2.20 GHz


### 3. Extracting or copying your videos/photos to desired folder

In [18]:
#%% (1) Extracting or copying your videos/photos to desired folder.
AI4GSD.getPhotos(PhotoDir,nframe=500, overwrite=False)                     # Use once.

Video photo output folder: /content/drive/MyDrive/AI4GSD/Demo_Data/20250521_YC_WalkSmartphoneRoadTest
Extracting IMG_9683.MOV every 500 frames.
IMG_9683.MOV: get frame 1 from 500/2171 in 0.00 s (23.03%), file existed, skipping...
IMG_9683.MOV: get frame 2 from 1000/2171 in 0.00 s (46.06%), file existed, skipping...
IMG_9683.MOV: get frame 3 from 1500/2171 in 0.00 s (69.09%), file existed, skipping...
IMG_9683.MOV: get frame 4 from 2000/2171 in 0.00 s (92.12%), file existed, skipping...

Computation done, elapsed time 0.25 seconds.




### 4. Obtaining scales from AI, drone flight data, or manual measurements

In [19]:
PP = AI4GSD.parameters()
PP.Directory = PhotoDir
PP.OverWriteAll = True                                                     # Overwrite everything if True.
PP.OverWriteScaleFile = False
PP.OverWriteScaleRows = True
PP.ScaleSource = 'AI'                                                      # Use AI option.
PP.ModelName = 'YOLO11m.1280.20250825'                                     # Which AI to use.
PP.ComputingSource = 'local'                                               # What computing resource to use.
PP.ConfidenceThresholdUser = 0.85                                          # Minimum confidence threshold.
PP.YOLOSave = False                                                        # Need to save AI prediction to folder.
scales =  AI4GSD.metadata(PP)

Generating scale file for: 20250521_YC_WalkSmartphoneRoadTest
Estimating photo resolution via local AI model: YOLO11m.1280.20250825
Model layer: 113, model parmeter: 20.06M, submodule number: 409
IOU: 0.45, user confidence: 0.85
1/4: photo 1/4, IMG_9683_500.png, scale greencap, res 0.62 mm/px, status 200, (25.00%), 6.50 s
2/4: photo 2/4, IMG_9683_1000.png, scale greencap, res 0.59 mm/px, status 200, (50.00%), 7.93 s
3/4: photo 3/4, IMG_9683_1500.png, scale greencap, res 0.59 mm/px, status 200, (75.00%), 4.30 s
4/4: photo 4/4, IMG_9683_2000.png, scale greencap, res 0.63 mm/px, status 200, (100.00%), 4.20 s
Total time: 23.18 s, time per photo 5.80 s
-----------------------------------------------------------------



### 5. Obtaining segmentation of anthropogenic objects

In [20]:
PP = AI4GSD.parameters()
PP.Directory = PhotoDir
PP.FolderLevel = 'L1'
PP.ModelName = 'YOLO11m.640.20250906'
PP.ObjectName = 'anthro'
PP.OverWriteSegementation = True
PP.YOLOSave = False
PP.YOLOIOU = 0.45
PP.ComputingSource = 'local'
PP.ImageSizeType = 'default'
PP.SegmentationClassName =['people','road','steel']                        # What object class need to track.
AI4GSD.background(PP)

-----------------------------------------------------------------
Folder name: 20250521_YC_WalkSmartphoneRoadTest
output name: segments
Model name: YOLO11m.640.20250906 from local for anthro
Model layer: 125, model parmeter: 22.37M, submodule number: 445
IOU: 0.45, user confidence: 0.35
Number of photos: 4, number of exluded photos 0
20250521_YC_WalkSmartphoneRoadTest: 1/4: IMG_9683_500.png (25.00%)
20250521_YC_WalkSmartphoneRoadTest: 2/4: IMG_9683_1000.png (50.00%)
20250521_YC_WalkSmartphoneRoadTest: 3/4: IMG_9683_1500.png (75.00%)
20250521_YC_WalkSmartphoneRoadTest: 4/4: IMG_9683_2000.png (100.00%)
Total time: 12.61 s, time per photo 3.15 s
-----------------------------------------------------------------



### 6. Obtaining grain size data

In [12]:
PP = AI4GSD.parameters()
PP.Directory = PhotoDir
PP.ModelName = 'YOLO11m.1280.20250322'
PP.YOLOSave = False
PP.SaveOverlayLabel = True
PP.OverWriteAll = True
PP.PrintOnScreen = True
PP.MaximumGrainSize = 0.15
PP.OverlayFontSize = 4
PP.OverlayLegendFontSize = 4
PP.OverlayGSDTickLineWidth = 4
gsd = AI4GSD.Photo2GSD(PP)

Output hidden; open in https://colab.research.google.com to view.

### 7. Applying quality control

In [21]:
gsdqc, stqc = AI4GSD.qualityControl(PhotoDir)


Quality control summary for 20250521_YC_WalkSmartphoneRoadTest:
1/1: IMG_9683.MOV, time 36.2 s, area 3.0 m2, valid area 2.2 m2 (73.3%), frame 4, valid frame 3 (75.0%)
1/1: ***, grain number 1840, number per image 460.0; efficiency 0.1 m2/s, 50.8 grains/s
